# Step 9: Photometric Redshifts

Weak lensing requires knowing the **distances** to galaxies (to assign
them to redshift bins). Spectroscopy is too slow for billions of galaxies,
so we estimate redshifts from **multi-band photometry** — "photo-z".

**Tool:** RAIL (Redshift Assessment Infrastructure Layers)  
**Input:** CModel fluxes in ugrizy bands  
**Output:** Redshift PDF $p(z)$ for each galaxy

**References:**
- RAIL: [github.com/LSSTDESC/rail](https://github.com/LSSTDESC/rail)
- Photo-z requirements: Mandelbaum (2018) §5

## 9.1 Why Photo-z Matters for Weak Lensing

The lensing signal depends on the **geometry** of the lens-source system:

$$\gamma \propto \Sigma_{\text{crit}}^{-1} = \frac{4\pi G}{c^2} \cdot \frac{D_{LS} D_L}{D_S}$$

where $D_L$, $D_S$, $D_{LS}$ are angular diameter distances to the lens,
source, and between them.

If we misestimate $z_{\text{source}}$, we get the wrong $\Sigma_{\text{crit}}$
and therefore the wrong mass for the lens. For cosmic shear, photo-z errors
bias the dark energy equation of state $w$ — LSST requires:

- **Bias:** $|\Delta z| < 0.003(1+z)$ per tomographic bin
- **Scatter:** $\sigma_z < 0.05(1+z)$
- **Outlier rate:** < 10%

## 9.2 How Photo-z Works

### The physical basis
Galaxy spectra have features (Balmer/Lyman breaks, emission lines) that
shift with redshift. Multi-band photometry samples this spectrum coarsely:

```
Spectrum:  ━━━━━╱╲━━━━━━━━━━━━━━━━━━━━━━
                 4000Å break

At z=0:   u    g    r    i    z    y     ← break falls in u-g
At z=0.5: u    g    r    i    z    y     ← break falls in g-r  
At z=1.0: u    g    r    i    z    y     ← break falls in r-i
```

The **colour** (flux ratio between bands) tells you which bands straddle
the break, and therefore the redshift.

### Template-fitting methods
1. Start with a library of galaxy SED templates (elliptical, spiral, starburst, ...)
2. For each template and each trial redshift, predict the ugrizy fluxes
3. Compare predicted to observed fluxes → $\chi^2(z)$ → $p(z)$

Examples: BPZ, EAZY, LePhare

### Machine learning methods
1. Train on a set of galaxies with known spectroscopic redshifts
2. Learn the mapping: (u, g, r, i, z, y) → z
3. Predict $p(z)$ for the full photometric sample

Examples: FlexZBoost, GPz, ANNz2, neural networks

### RAIL unifies both approaches
RAIL provides a common API for all photo-z methods:
```python
from rail.estimation.algos.bpz_lite import BPZliteEstimator
# or
from rail.estimation.algos.flexzboost import FlexZBoostEstimator
# Same interface:
estimator = BPZliteEstimator.make_stage(name='bpz', ...)
results = estimator.estimate(test_data)
# results.data contains p(z) on a grid for each galaxy
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

In [ ]:
# --- Simulate how galaxy colours change with redshift ---

# Simplified: model the 4000Å break moving through LSST bands
# Band effective wavelengths (Å)
bands = {'u': 3671, 'g': 4827, 'r': 6223, 'i': 7546, 'z': 8691, 'y': 9712}
band_names = list(bands.keys())
band_waves = np.array(list(bands.values()))

def galaxy_sed(wavelength, z, galaxy_type='elliptical'):
    """Simplified galaxy SED with a 4000Å break."""
    rest_wave = wavelength / (1 + z)
    break_wave = 4000.0  # Å
    break_strength = 0.5 if galaxy_type == 'elliptical' else 0.2

    # Power-law continuum with break
    sed = (rest_wave / 5000)**(-1.5)
    sed[rest_wave < break_wave] *= (1 - break_strength)
    return sed


# Compute colours as a function of redshift
z_grid = np.linspace(0, 2.5, 100)
colours = {f'{b1}-{b2}': [] for b1, b2 in zip(band_names[:-1], band_names[1:])}

for z in z_grid:
    fluxes = galaxy_sed(band_waves, z, 'elliptical')
    mags = -2.5 * np.log10(fluxes + 1e-30)
    for i, (b1, b2) in enumerate(zip(band_names[:-1], band_names[1:])):
        colour = mags[i] - mags[i+1]
        colours[f'{b1}-{b2}'].append(colour)

fig, ax = plt.subplots(figsize=(10, 6))
color_cycle = ['purple', 'blue', 'green', 'orange', 'red']
for (name, vals), c in zip(colours.items(), color_cycle):
    ax.plot(z_grid, vals, lw=2, color=c, label=name)

ax.set_xlabel('Redshift z', fontsize=12)
ax.set_ylabel('Colour (mag)', fontsize=12)
ax.set_title('Galaxy Colours vs. Redshift (elliptical SED)\n'
             'The 4000Å break moves through successive band pairs',
             fontsize=13)
ax.legend(fontsize=11, ncol=2)
ax.set_xlim(0, 2.5)
plt.tight_layout()
plt.savefig('../../figures/colour_redshift.png', dpi=150, bbox_inches='tight')
plt.show()

print("Each colour becomes red when the break enters that band pair.")
print("This is the physical basis for photo-z estimation.")

In [ ]:
# --- Simulate photo-z estimation (template fitting) ---

n_galaxies = 200
z_true = rng.uniform(0.1, 2.0, n_galaxies)
gal_types = rng.choice(['elliptical', 'spiral'], n_galaxies, p=[0.4, 0.6])

# "Observe" each galaxy
observed_fluxes = []
for z, gtype in zip(z_true, gal_types):
    true_flux = galaxy_sed(band_waves, z, gtype)
    # Add realistic noise (fainter galaxies are noisier)
    noise = 0.05 * true_flux * rng.normal(size=len(band_waves))
    observed_fluxes.append(true_flux + noise)

observed_fluxes = np.array(observed_fluxes)

# Template fit: for each galaxy, find best-fit z by chi2 minimization
z_trial = np.linspace(0, 2.5, 200)
z_photo = []

for i in range(n_galaxies):
    chi2 = []
    for zt in z_trial:
        for gtype in ['elliptical', 'spiral']:
            model = galaxy_sed(band_waves, zt, gtype)
            # Scale model to match observed flux
            scale = np.sum(observed_fluxes[i] * model) / np.sum(model**2)
            resid = observed_fluxes[i] - scale * model
            noise_est = 0.05 * np.abs(observed_fluxes[i]) + 1e-10
            chi2.append(np.sum(resid**2 / noise_est**2))

    # Reshape: (n_z_trial, n_types) and take minimum over types
    chi2 = np.array(chi2).reshape(len(z_trial), 2).min(axis=1)
    z_photo.append(z_trial[np.argmin(chi2)])

z_photo = np.array(z_photo)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Photo-z vs spec-z
ax = axes[0]
ax.scatter(z_true, z_photo, s=10, alpha=0.5, c='steelblue')
ax.plot([0, 2.5], [0, 2.5], 'r--', lw=1.5, label='z_phot = z_true')
ax.set_xlabel('True redshift $z_{\\rm true}$', fontsize=12)
ax.set_ylabel('Photo-z $z_{\\rm phot}$', fontsize=12)
ax.set_title('Photometric vs. True Redshift', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 2.2); ax.set_ylim(0, 2.5)
ax.set_aspect('equal')

# Residual distribution
ax = axes[1]
dz = (z_photo - z_true) / (1 + z_true)
ax.hist(dz, bins=40, range=(-0.2, 0.2), color='steelblue', edgecolor='white')
ax.axvline(0, color='red', ls='--', lw=1.5)
ax.set_xlabel('$\\Delta z / (1 + z_{\\rm true})$', fontsize=12)
ax.set_ylabel('Count', fontsize=12)

bias = np.median(dz)
scatter = 1.4826 * np.median(np.abs(dz - np.median(dz)))  # robust σ
outlier_frac = np.mean(np.abs(dz) > 0.15)
ax.set_title(f'Photo-z residuals\nbias={bias:.4f}, σ={scatter:.4f}, outliers={outlier_frac:.1%}',
             fontsize=13)

plt.tight_layout()
plt.savefig('../../figures/photoz_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"LSST requirements: |bias| < 0.003, σ < 0.05, outliers < 10%")
print(f"Our toy model:     |bias| = {abs(bias):.4f}, σ = {scatter:.4f}, outliers = {outlier_frac:.1%}")

## 9.3 Tomographic Binning

For cosmic shear, galaxies are sorted into **redshift bins** (tomographic bins).
The shear correlation function is measured within and between bins:

```
Bin 1: 0.2 < z < 0.5    ┐
Bin 2: 0.5 < z < 0.8    ├── auto- and cross-correlations
Bin 3: 0.8 < z < 1.1    │   give cosmological constraints
Bin 4: 1.1 < z < 1.5    ┘
```

Photo-z errors cause galaxies to scatter between bins, diluting the
signal and introducing biases. This is why photo-z calibration is
one of the dominant systematics for LSST weak lensing.

## Summary

| Component | Method | Input | Output |
|-----------|--------|-------|--------|
| Photo-z | Template fitting or ML | ugrizy fluxes | p(z) per galaxy |
| Tomography | Binning by z_phot | p(z) | Source sample in z bins |
| Calibration | Cross-correlation with spec-z | Spec-z training set | True n(z) per bin |

---

## Congratulations!

You've walked through the entire LSST pipeline from raw CCD pixels
to a weak lensing shear catalog with photometric redshifts:

```
Raw CCD → ISR → Background + PSF → Calibration → Coadd →
Detection → Deblending → Shape (HSM) + Flux (CModel) → Photo-z → Shear catalog
```

### Next steps for the project:
1. **`notebooks/03_shapes/`** — Deep dive into shape measurement on real HSC data
2. Work through the [DP0.2 tutorials](https://dp0-2.lsst.io/) on the Rubin Science Platform
3. Run the actual LSST pipeline on HSC data with `lsst.meas.extensions.shapeHSM`